# LA1 — prithvi: ton
Attach the FULL completed preprocessing outputs. Show original sources, protocol, quarantine and five-plus development plots.
Source code and dependencies are revision-pinned by the bootstrap.
See experiments/public_dataset_la/DATASET_GUIDE.md for original testbed and modality details.

In [ ]:
import sys, subprocess, shutil, uuid, time
from pathlib import Path
print('START repository bootstrap', flush=True)
started = time.monotonic()
assert Path('/kaggle/working').exists(), 'Execute scientific notebooks on Kaggle'
REPOSITORY = 'https://github.com/saltypal/DDoS_Prevention_and_Mitigation.git'
REVISION = None  # None resolves latest main ONCE. Set a recorded SHA to reproduce a run.
checkout = Path('/kaggle/temp') / ('ddos-source-' + uuid.uuid4().hex)
subprocess.run(['git','clone','--depth','1',REPOSITORY,str(checkout)],check=True,timeout=180)
if REVISION:
    subprocess.run(['git','fetch','--depth','1','origin',REVISION],cwd=checkout,check=True,timeout=180)
    subprocess.run(['git','checkout','--detach','FETCH_HEAD'],cwd=checkout,check=True)
SOURCE_COMMIT = subprocess.check_output(['git','rev-parse','HEAD'],cwd=checkout,text=True).strip()
print('PINNED SOURCE',SOURCE_COMMIT,checkout,flush=True)
if not shutil.which('java'):
    subprocess.run(['apt-get','update','-qq'],check=True,timeout=300)
    subprocess.run(['apt-get','install','-y','openjdk-17-jre-headless'],check=True,timeout=600)
subprocess.run(['java','-version'],check=True)
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check','-c',str(checkout/'experiments/public_dataset_la/constraints-kaggle.txt'),'-e',str(checkout)+'[spark,boost,notebooks]'],check=True,timeout=900)
sys.path.insert(0,str(checkout))
import os
os.chdir(checkout)
print('END repository bootstrap',time.monotonic()-started,'seconds',flush=True)
from sdn_ddos.runtime.notebook import install_capture
capture_root=Path('/kaggle/working/experiments/public_dataset_la/cell_capture') / uuid.uuid4().hex
install_capture(capture_root)


In [ ]:
from IPython.display import display, Markdown
display(Markdown((checkout/'experiments/public_dataset_la/DATASET_GUIDE.md').read_text()))

In [ ]:
from sdn_ddos.analysis.presentation import show_la1
print('START LA1',flush=True)
REBUILD=False  # True reruns the same source sanitation/splits/EDA modules on Kaggle.
artifact_root='/kaggle/input'
if REBUILD:
    import json
    from sdn_ddos.cli import execute
    config=json.loads((checkout/'experiments/public_dataset_la/configs/prepare.json').read_text())
    config['sources']=['ton']
    config['name']='la1-prithvi-ton'
    artifact_root=str(execute(config))
show_la1('ton',root=artifact_root)
print('END LA1',flush=True)
